# Rare3D-FM 服务器训练入口

选择服务器上的 `rare3d` Python 内核，然后从上到下运行。
本 Notebook 只负责配置和启动，模型、数据处理、训练及重建保留在 `rare3d/` 模块中。

- 数据目录：项目根目录下的 `data/Zhuang-ABCA-2/`。
- 使用物理 **GPU 1**；屏蔽其他显卡后，PyTorch 中对应 `cuda:0`。
- 默认执行完整训练；需要先检查流程时，将下方 `SMOKE_TEST` 改为 `True`。
- GPU 设置单元必须在导入 PyTorch 和初始化 CUDA 前执行。


In [ ]:
import os
import sys

# 已在其他单元用不同设置初始化 CUDA 时，必须重启内核再从这里执行。
if ("torch" in sys.modules
        and sys.modules["torch"].cuda.is_initialized()
        and (os.environ.get("CUDA_VISIBLE_DEVICES") != "1"
             or os.environ.get("CUDA_DEVICE_ORDER") != "PCI_BUS_ID")):
    raise RuntimeError("CUDA 已使用其他 GPU 设置初始化，请重启内核并从第一个代码单元运行。")

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
print("CUDA_VISIBLE_DEVICES=1：使用物理 GPU 1（进程内 cuda:0）")


## 1. 路径与训练参数

从项目根目录打开此 Notebook。数据目录中应直接包含 `sec_*.h5ad` 切片。
`SMOKE_TEST=False` 使用 `configs/local.json` 的完整训练轮数；试跑会自动使用独立输出目录。


In [ ]:
from pathlib import Path
import json

PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / "rare3d").is_dir():
    raise RuntimeError("请将 Notebook 的工作目录设为项目根目录（包含 rare3d/ 和 configs/）。")

from rare3d.config import load_config, smoke_config, validate_config

SMOKE_TEST = False
DATA_DIR = PROJECT_DIR / "data" / "Zhuang-ABCA-2"
OUTPUT_DIR = PROJECT_DIR / "outputs" / ("server_smoke_gpu1" if SMOKE_TEST else "server_gpu1")

cfg = load_config()
if SMOKE_TEST:
    cfg = smoke_config(cfg)
cfg.update(data_dir=str(DATA_DIR), output_dir=str(OUTPUT_DIR), device="cuda")
validate_config(cfg)
print(json.dumps(cfg, indent=2, ensure_ascii=False))


## 2. 确认 GPU 和切片

此处检查 CUDA 实际可用且能够执行计算，并列出待加载的切片。


In [ ]:
import torch
from rare3d.data import find_h5ad_files, select_files

if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用，请确认当前 Notebook 内核为安装了 CUDA 版 PyTorch 的 rare3d 环境。")
torch.cuda.set_device(0)
print("PyTorch:", torch.__version__)
print("CUDA runtime:", torch.version.cuda)
print("物理 GPU 1 / 进程内 cuda:0:", torch.cuda.get_device_name(0))
probe = torch.ones(1, device="cuda:0")
print("GPU 计算检查:", (probe + 1).item())
del probe

h5_files = select_files(find_h5ad_files(cfg["data_dir"]), cfg["max_sections"])
print(f"数据目录: {DATA_DIR}")
print(f"切片数: {len(h5_files)}")
print("前几个切片:", [p.name for p in h5_files[:5]])


## 3. 训练、验证与导出

依次执行 VAE、稀有原型发现、微环境训练、流匹配训练、验证和三维重建导出。
完整训练耗时较长；重新执行此单元会从头训练，并覆盖同一输出目录中的结果。


In [ ]:
from rare3d.pipeline import run

result_dir = run(cfg)
print(f"结果目录: {result_dir}")


## 4. 查看验证结果

小规模试跑仅验证流程，不能作为最终模型质量的结论。


In [ ]:
from IPython.display import Image, display

metrics_path = result_dir / "metrics.json"
print(metrics_path.read_text())
figure_path = result_dir / "rare3d_val_comparison.png"
if figure_path.is_file():
    display(Image(filename=str(figure_path)))
print("已保存文件:")
for path in sorted(result_dir.iterdir()):
    if path.is_file():
        print(f"  {path.name} ({path.stat().st_size / 1024**2:.2f} MiB)")
